# 02 - Data Preparation and Integration  
  
*AFL Matchday Demand FOrecasting - Preparing Reliable Match and Calendar Tables*

## 1. Objective and Scope

### 1.1 Notebook Objective

This notebook prepares the validated raw datasets.

It transforms source files with different formats, field names, and naming conventions into consistent, database-ready tables for later PostgreSQL loading and feature engineering.

The preparation focuses on three data groups:

- historical AFL match and attendance data;
- 2026 Squiggle match and team snapshots;
- historical and official school-holiday records.

The original source files remain unchanged. All transformations are applied to in-memory copies, and the validated results are exported separately.

### 1.2 Selected v1 Data Sources

Only data sources that directly support match-level demand forecasting or the preparation of 2026 scoring fixtures are included in version 1.

| Data source | Raw format | Purpose in this notebook |
|---|---|---|
| AFLStats `games.csv` | CSV | Provides historical match-level results, attendance, teams, venues, and round information. |
| `squiggle_games_2026_20260817.json` | JSON | Provides the 2026 match and fixture records available at the snapshot date. |
| `squiggle_teams_20260817.json` | JSON | Provides Squiggle team identifiers and names for consistent team mapping. |
| Historical school-holiday dataset from Figshare | R data file | Provides daily city-level school-holiday records for 2004–2023. |
| Official state and territory school calendars | PDF documents | Provide reviewed school-term and holiday periods for 2024–2026. |

The downloaded `players.csv` and `stats.csv` files are retained in the raw-data directory but are not processed in this notebook. Version 1 uses match-level predictors and does not include player-level features.

### 1.3 Expected Outputs and Processing Boundaries

This notebook prepares five structured datasets for later database loading and feature engineering.

| Prepared output | Record grain | Intended use |
|---|---|---|
| Historical match table | One row per completed historical match | Supports later feature engineering and model development. |
| Prepared 2026 match snapshot | One row per Squiggle match or scheduled fixture | Separates completed matches from future scoring fixtures. |
| Team reference table | One row per standardised AFL team | Provides consistent team names and identifiers across sources. |
| Venue reference table | One row per standardised venue | Provides consistent venue names and associated locations. |
| Daily school-holiday table | One row per covered city and date | Supports later calendar-feature creation through match date and venue location. |

Validated outputs will be exported to `data/interim/postgres_ready/`.

The processing included in this notebook is limited to:

- selecting required source fields;
- standardising column names and data types;
- aligning team and venue names;
- applying previously reviewed source corrections;
- structuring and combining school-holiday records;
- validating key fields, identifiers, mappings, and date coverage;
- exporting the prepared tables.

The following tasks are outside the scope of this notebook:

- processing player-level data;
- creating predictive features or attendance classes;
- splitting data into training, validation, and test periods;
- training or evaluating machine-learning models;
- loading data into PostgreSQL;
- collecting live weather forecasts;
- deploying or automating the workflow in AWS.

## 2. Load the Selected Data Sources

### 2.1 Historical Match Data

The historical match dataset is loaded from the AFLStats `games.csv` file reviewed in Notebook 01.

At this stage, the notebook confirms that the source can be loaded successfully and that the fields required for later preparation are available. The detailed raw-data audit is not repeated here.

**Load and initial confirmation**

The source CSV is loaded using a project-relative path so that the notebook does not depend on a specific local drive or user directory. Only a small preview and structural summary are displayed at this stage.

In [1]:
from pathlib import Path

import pandas as pd


# Identify the project root so the notebook remains portable when it is
# executed from either the project directory or the notebooks directory.
notebook_working_directory = Path.cwd()
project_root = (
    notebook_working_directory.parent
    if notebook_working_directory.name.lower() == "notebooks"
    else notebook_working_directory
)

# Construct the source path relative to the project root rather than
# relying on a machine-specific absolute path.
historical_match_path = (
    project_root
    / "data"
    / "raw"
    / "extracted"
    / "aflstats_v6"
    / "games.csv"
)

# Stop execution with a clear message if the expected audited source
# is unavailable or has been moved.
if not historical_match_path.is_file():
    raise FileNotFoundError(
        f"Historical match source was not found: {historical_match_path}"
    )

# Load the audited source into memory without modifying the original CSV.
# Disabling chunk-based type inference provides consistent initial dtypes.
historical_matches_raw = pd.read_csv(
    historical_match_path,
    low_memory=False,
)

# Display a concise structural summary for reproducibility and review.
print(f"Source: {historical_match_path.relative_to(project_root)}")
print(f"Rows: {len(historical_matches_raw):,}")
print(f"Columns: {historical_matches_raw.shape[1]}")
print("\nColumn names:")
print(historical_matches_raw.columns.tolist())

display(historical_matches_raw.head(3))

Source: data\raw\extracted\aflstats_v6\games.csv
Rows: 2,879
Columns: 22

Column names:
['GameId', 'Year', 'Round', 'Date', 'MaxTemp', 'MinTemp', 'Rainfall', 'Venue', 'StartTime', 'Attendance', 'HomeTeam', 'HomeTeamScoreQT', 'HomeTeamScoreHT', 'HomeTeamScore3QT', 'HomeTeamScoreFT', 'HomeTeamScore', 'AwayTeam', 'AwayTeamScoreQT', 'AwayTeamScoreHT', 'AwayTeamScore3QT', 'AwayTeamScoreFT', 'AwayTeamScore']


,GameId,Year,Round,Date,MaxTemp,MinTemp,Rainfall,Venue,StartTime,Attendance,...,HomeTeamScoreHT,HomeTeamScore3QT,HomeTeamScoreFT,HomeTeamScore,AwayTeam,AwayTeamScoreQT,AwayTeamScoreHT,AwayTeamScore3QT,AwayTeamScoreFT,AwayTeamScore
0,2012R0101,2012,Round 1,2012-03-24,24.0,12.2,0.0,Stadium Australia,7:20 PM,"38,203",...,3.3,3.4,5.70,37,Sydney,4.1,8.4,13.80,14.16,100
1,2012R0102,2012,Round 1,2012-03-29,25.7,9.7,0.0,MCG,7:45 PM,"78,285",...,5.6,10.7,12.90,81,Carlton,3.2,8.7,11.13,18.17,125
2,2012R0103,2012,Round 1,2012-03-30,27.4,9.7,0.6,MCG,7:50 PM,"78,466",...,10.6,14.1,20.17,137,Collingwood,2.7,7.9,12.16,16.19,115


**Required-field confirmation**

The following check confirms that the fields required for match-level preparation are present in the source schema. Optional source fields are retained in the loaded DataFrame but are not treated as required version 1 inputs.

In [2]:
# Define the source fields required for later match-level preparation.
# Quarter-by-quarter scores and weather observations remain available
# in the raw DataFrame but are not required by this loading check.
required_historical_columns = [
    "GameId",
    "Year",
    "Round",
    "Date",
    "Venue",
    "StartTime",
    "Attendance",
    "HomeTeam",
    "AwayTeam",
    "HomeTeamScore",
    "AwayTeamScore",
]

# Build a readable schema check before applying any transformations.
required_column_check = pd.DataFrame(
    {
        "column_name": required_historical_columns,
        "present_in_source": [
            column in historical_matches_raw.columns
            for column in required_historical_columns
        ],
        "source_dtype": [
            (
                str(historical_matches_raw[column].dtype)
                if column in historical_matches_raw.columns
                else None
            )
            for column in required_historical_columns
        ],
    }
)

display(required_column_check)

# Stop the workflow if a required field is missing, as subsequent
# preparation steps would otherwise produce misleading failures.
missing_required_columns = required_column_check.loc[
    ~required_column_check["present_in_source"],
    "column_name",
].tolist()

if missing_required_columns:
    raise ValueError(
        "Missing required historical match columns: "
        f"{missing_required_columns}"
    )

print("Required historical column check: PASSED")

,column_name,present_in_source,source_dtype
0,GameId,True,str
1,Year,True,int64
2,Round,True,str
3,Date,True,str
4,Venue,True,str
5,StartTime,True,str
6,Attendance,True,str
7,HomeTeam,True,str
8,AwayTeam,True,str
9,HomeTeamScore,True,int64


Required historical column check: PASSED


**Result**

The historical match source loaded successfully with 2,879 rows and 22 source columns. All 11 fields required for later match-level preparation are present.

`Date`, `StartTime`, and `Attendance` are currently stored as object-type values. Their conversion is intentionally deferred to Section 3.1, where column names and data types will be standardised together.

No source records or values were modified in this section.

### 2.2 Squiggle Games and Teams Snapshots

The Squiggle API snapshots capture the 2026 game records and team reference information available on 17 August 2026.

The games snapshot may contain both completed matches and scheduled fixtures. The teams snapshot provides the source identifiers and names needed to interpret those game records consistently.

This section loads the two saved JSON responses and confirms their basic structure. Match-status classification and name standardisation are deferred to Section 4.

**Load and verify the saved API responses**

The saved responses are loaded as JSON objects before their record collections are converted into DataFrames. Checking the expected top-level keys first makes any unexpected change in the response structure easier to identify.

In [3]:
import json


# Define both snapshot paths relative to the previously identified
# project root so that the notebook remains portable.
squiggle_games_path = (
    project_root
    / "data"
    / "raw"
    / "api_snapshots"
    / "squiggle_games_2026_20260817.json"
)

squiggle_teams_path = (
    project_root
    / "data"
    / "raw"
    / "api_snapshots"
    / "squiggle_teams_20260817.json"
)

# Confirm that both saved API responses are available before loading
# either file, preventing a partially loaded snapshot pair.
snapshot_paths = {
    "games": squiggle_games_path,
    "teams": squiggle_teams_path,
}

missing_snapshot_files = [
    name
    for name, path in snapshot_paths.items()
    if not path.is_file()
]

if missing_snapshot_files:
    raise FileNotFoundError(
        "Missing Squiggle snapshot files: "
        f"{missing_snapshot_files}"
    )

# Load the complete JSON payloads so their original response structure
# remains available for inspection.
with squiggle_games_path.open(
    mode="r",
    encoding="utf-8",
) as file:
    squiggle_games_payload = json.load(file)

with squiggle_teams_path.open(
    mode="r",
    encoding="utf-8",
) as file:
    squiggle_teams_payload = json.load(file)

# Validate the expected Squiggle response collections before
# converting their records into tabular form.
if not isinstance(squiggle_games_payload.get("games"), list):
    raise ValueError(
        "The games snapshot does not contain an expected 'games' list."
    )

if not isinstance(squiggle_teams_payload.get("teams"), list):
    raise ValueError(
        "The teams snapshot does not contain an expected 'teams' list."
    )

# Convert the record collections into DataFrames without changing
# field names or source values.
squiggle_games_raw = pd.json_normalize(
    squiggle_games_payload["games"]
)

squiggle_teams_raw = pd.json_normalize(
    squiggle_teams_payload["teams"]
)

# Report the loaded structures without performing preparation yet.
print(
    "Games source:",
    squiggle_games_path.relative_to(project_root),
)
print(
    f"Games records: {len(squiggle_games_raw):,}; "
    f"columns: {squiggle_games_raw.shape[1]}"
)
print("Games columns:")
print(squiggle_games_raw.columns.tolist())

print(
    "\nTeams source:",
    squiggle_teams_path.relative_to(project_root),
)
print(
    f"Team records: {len(squiggle_teams_raw):,}; "
    f"columns: {squiggle_teams_raw.shape[1]}"
)
print("Teams columns:")
print(squiggle_teams_raw.columns.tolist())

Games source: data\raw\api_snapshots\squiggle_games_2026_20260817.json
Games records: 218; columns: 26
Games columns:
['hscore', 'ateamid', 'hgoals', 'is_final', 'complete', 'date', 'winner', 'agoals', 'unixtime', 'ascore', 'hteamid', 'tz', 'timestr', 'updated', 'hbehinds', 'ateam', 'localtime', 'winnerteamid', 'is_grand_final', 'round', 'year', 'roundname', 'hteam', 'id', 'abehinds', 'venue']

Teams source: data\raw\api_snapshots\squiggle_teams_20260817.json
Team records: 18; columns: 6
Teams columns:
['retirement', 'name', 'id', 'abbrev', 'debut', 'logo']


**Required-field confirmation**

Squiggle defines `complete` as the estimated percentage of a match completed. In contrast, `is_final` identifies whether the match belongs to the finals series and, where applicable, its finals category.

Completion status will therefore be derived from `complete`, not from `is_final`, during the preparation stage.

The following checks confirm that the match, scheduling, team, score, and status fields required for later processing are present in both snapshots.

Source: [Squiggle API documentation](https://api.squiggle.com.au/)

In [4]:
# Define the Squiggle game fields required for fixture preparation,
# team mapping, time standardisation, and completion classification.
required_squiggle_game_columns = [
    "id",
    "year",
    "round",
    "date",
    "localtime",
    "tz",
    "hteamid",
    "hteam",
    "ateamid",
    "ateam",
    "venue",
    "complete",
    "is_final",
    "hscore",
    "ascore",
]

# Define the team-reference fields required to interpret the team
# identifiers and names contained in the games snapshot.
required_squiggle_team_columns = [
    "id",
    "name",
    "abbrev",
    "debut",
    "retirement",
]

# Identify missing fields before displaying or using either schema.
missing_game_columns = [
    column
    for column in required_squiggle_game_columns
    if column not in squiggle_games_raw.columns
]

missing_team_columns = [
    column
    for column in required_squiggle_team_columns
    if column not in squiggle_teams_raw.columns
]

# Stop the workflow with a source-specific message if either saved
# response does not contain the required fields.
if missing_game_columns:
    raise ValueError(
        "Missing required Squiggle game columns: "
        f"{missing_game_columns}"
    )

if missing_team_columns:
    raise ValueError(
        "Missing required Squiggle team columns: "
        f"{missing_team_columns}"
    )

# Create readable schema summaries for the required fields only.
squiggle_game_column_check = pd.DataFrame(
    {
        "column_name": required_squiggle_game_columns,
        "present_in_source": True,
        "source_dtype": [
            str(squiggle_games_raw[column].dtype)
            for column in required_squiggle_game_columns
        ],
    }
)

squiggle_team_column_check = pd.DataFrame(
    {
        "column_name": required_squiggle_team_columns,
        "present_in_source": True,
        "source_dtype": [
            str(squiggle_teams_raw[column].dtype)
            for column in required_squiggle_team_columns
        ],
    }
)

print("Required Squiggle game fields:")
display(squiggle_game_column_check)

print("Required Squiggle team fields:")
display(squiggle_team_column_check)

print("Required Squiggle column checks: PASSED")

Required Squiggle game fields:


,column_name,present_in_source,source_dtype
0,id,True,int64
1,year,True,int64
2,round,True,int64
3,date,True,str
4,localtime,True,str
5,tz,True,str
6,hteamid,True,float64
7,hteam,True,str
8,ateamid,True,float64
9,ateam,True,str


Required Squiggle team fields:


,column_name,present_in_source,source_dtype
0,id,True,int64
1,name,True,str
2,abbrev,True,str
3,debut,True,int64
4,retirement,True,int64


Required Squiggle column checks: PASSED


**Compact source preview**

A limited set of fields is displayed to confirm the meaning and alignment of the two API responses without producing an unnecessarily wide notebook output. No rows are filtered or transformed.

In [5]:
# Select a compact set of game fields that represents identity,
# scheduling, participating teams, venue, and source status.
squiggle_game_preview_columns = [
    "id",
    "year",
    "round",
    "date",
    "hteam",
    "ateam",
    "venue",
    "complete",
    "is_final",
]

# Select the team fields needed to interpret the source identifiers
# used by the games snapshot.
squiggle_team_preview_columns = [
    "id",
    "name",
    "abbrev",
    "debut",
    "retirement",
]

print("Squiggle games snapshot preview:")
display(
    squiggle_games_raw.loc[
        :,
        squiggle_game_preview_columns,
    ].head(3)
)

print("Squiggle teams snapshot preview:")
display(
    squiggle_teams_raw.loc[
        :,
        squiggle_team_preview_columns,
    ].head(3)
)

Squiggle games snapshot preview:


,id,year,round,date,hteam,ateam,venue,complete,is_final
0,38494,2026,0,2026-03-05 19:30:00,Sydney,Carlton,S.C.G.,100,0
1,38495,2026,0,2026-03-06 20:05:00,Gold Coast,Geelong,Carrara,100,0
2,38496,2026,0,2026-03-07 16:15:00,Greater Western Sydney,Hawthorn,Sydney Showground,100,0


Squiggle teams snapshot preview:


,id,name,abbrev,debut,retirement
0,1,Adelaide,ADE,1991,9999
1,2,Brisbane Lions,BRI,1987,9999
2,3,Carlton,CAR,1897,9999


**Result**

The saved Squiggle games response loaded successfully with 218 records and 26 source fields. The teams response loaded successfully with 18 records and 6 source fields. All fields required for later preparation are present.

The games snapshot provides match identifiers, scheduling details, participating teams, venues, scores, completion progress, and finals categories. The teams snapshot provides the source identifiers and names needed to interpret the team fields consistently.

No records were filtered, classified, or standardised in this section. Completed matches, confirmed future fixtures, and fixtures with teams not yet determined will be distinguished in Section 4.

### 2.3 Historical and Official School-Holiday Sources

The school-holiday inputs are stored in two different forms.

The historical Figshare source provides structured daily holiday records for 2004–2023. Official school-calendar documents from the relevant Australian state and territory authorities provide the source evidence needed to extend coverage through 2024–2026.

This section confirms that both source groups are available and loads only the existing structured historical data. The official PDF documents remain unchanged and are not treated as database-ready tables. Their reviewed date periods will be structured later in Section 5.

**Load the historical data and inventory the official documents**

An RDA file can contain one or more named R objects rather than a single visible table. It is therefore loaded as an object collection before the relevant DataFrame is selected.

The official calendar PDFs are inventoried but not parsed in this section. They remain source documents for the reviewed date-period preparation in Section 5.

In [6]:
import pyreadr


# Define the historical RDA source and the directory containing the
# official calendar documents using project-relative paths.
historical_holiday_path = (
    project_root
    / "data"
    / "raw"
    / "extracted"
    / "school_holidays_figshare_v1"
    / "school.holidays.rda"
)

official_calendar_directory = (
    project_root
    / "data"
    / "raw"
    / "source_documents"
    / "school_calendars_official_2024_2026"
)

# Define the official source documents expected from the completed
# collection step, including the two separate NSW documents.
expected_official_calendar_files = {
    "act_school_terms_2024_2026.pdf",
    "nsw_school_terms_2024_2025.pdf",
    "nsw_school_terms_2026_eastern.pdf",
    "nt_school_terms_2024_2026.pdf",
    "qld_school_terms_2024_2026.pdf",
    "sa_school_terms_2024_2026.pdf",
    "tas_school_terms_2024_2026.pdf",
    "vic_school_terms_2024_2026.pdf",
    "wa_school_terms_2024_2026.pdf",
}

# Confirm that the structured historical source and the official
# document directory are both available.
if not historical_holiday_path.is_file():
    raise FileNotFoundError(
        f"Historical holiday source was not found: "
        f"{historical_holiday_path}"
    )

if not official_calendar_directory.is_dir():
    raise FileNotFoundError(
        f"Official calendar directory was not found: "
        f"{official_calendar_directory}"
    )

# Inventory the available PDF files without extracting or changing
# their contents at this stage.
official_calendar_paths = sorted(
    official_calendar_directory.glob("*.pdf")
)

available_official_calendar_files = {
    path.name
    for path in official_calendar_paths
}

missing_official_calendar_files = sorted(
    expected_official_calendar_files
    - available_official_calendar_files
)

if missing_official_calendar_files:
    raise FileNotFoundError(
        "Missing official school-calendar documents: "
        f"{missing_official_calendar_files}"
    )

# Load the RDA container while preserving its original object names.
historical_holiday_objects = pyreadr.read_r(
    str(historical_holiday_path)
)

if not historical_holiday_objects:
    raise ValueError(
        "The historical school-holiday RDA file contains no readable objects."
    )

# Report the R objects and official documents available for later use.
print(
    "Historical source:",
    historical_holiday_path.relative_to(project_root),
)
print("R objects found:")

for object_name, object_data in historical_holiday_objects.items():
    print(
        f"- {object_name!r}: "
        f"{object_data.shape[0]:,} rows × "
        f"{object_data.shape[1]} columns"
    )

print(
    "\nOfficial calendar directory:",
    official_calendar_directory.relative_to(project_root),
)
print(f"Official PDF files found: {len(official_calendar_paths)}")

for calendar_path in official_calendar_paths:
    print(f"- {calendar_path.name}")

Historical source: data\raw\extracted\school_holidays_figshare_v1\school.holidays.rda
R objects found:
- 'school.holidays': 58,440 rows × 4 columns

Official calendar directory: data\raw\source_documents\school_calendars_official_2024_2026
Official PDF files found: 9
- act_school_terms_2024_2026.pdf
- nsw_school_terms_2024_2025.pdf
- nsw_school_terms_2026_eastern.pdf
- nt_school_terms_2024_2026.pdf
- qld_school_terms_2024_2026.pdf
- sa_school_terms_2024_2026.pdf
- tas_school_terms_2024_2026.pdf
- vic_school_terms_2024_2026.pdf
- wa_school_terms_2024_2026.pdf


**Select and preview the historical holiday table**

The RDA container includes the expected `school.holidays` object. A copy is assigned to a clearly named DataFrame so that subsequent preparation does not alter the object returned by the source reader.

The initial preview displays one complete city set for the first date, making the city–date record grain visible.

In [7]:
# Select the expected R object explicitly rather than relying on
# its position within the RDA container.
historical_holiday_object_name = "school.holidays"

if historical_holiday_object_name not in historical_holiday_objects:
    raise KeyError(
        "Expected R object was not found: "
        f"{historical_holiday_object_name}"
    )

# Work with an independent in-memory copy while preserving the
# source object returned by pyreadr.
historical_school_holidays_raw = historical_holiday_objects[
    historical_holiday_object_name
].copy()

# Define the complete source schema expected from the audited dataset.
required_historical_holiday_columns = [
    "City",
    "Date",
    "schoolhols",
    "school.hols",
]

missing_historical_holiday_columns = [
    column
    for column in required_historical_holiday_columns
    if column not in historical_school_holidays_raw.columns
]

if missing_historical_holiday_columns:
    raise ValueError(
        "Missing historical school-holiday columns: "
        f"{missing_historical_holiday_columns}"
    )

# Present the source data types before any standardisation is applied.
historical_holiday_column_check = pd.DataFrame(
    {
        "column_name": required_historical_holiday_columns,
        "present_in_source": True,
        "source_dtype": [
            str(historical_school_holidays_raw[column].dtype)
            for column in required_historical_holiday_columns
        ],
    }
)

display(historical_holiday_column_check)

# Display the first eight records because the source contains
# eight city observations for each calendar date.
display(
    historical_school_holidays_raw.loc[
        :,
        required_historical_holiday_columns,
    ].head(8)
)

print("Historical school-holiday column check: PASSED")

,column_name,present_in_source,source_dtype
0,City,True,str
1,Date,True,object
2,schoolhols,True,category
3,school.hols,True,category


,City,Date,schoolhols,school.hols
0,Adelaide,2004-01-01,1,4
1,Brisbane,2004-01-01,1,4
2,Canberra,2004-01-01,1,4
3,Darwin,2004-01-01,1,4
4,Hobart,2004-01-01,1,4
5,Melbourne,2004-01-01,1,4
6,Perth,2004-01-01,1,4
7,Sydney,2004-01-01,1,4


Historical school-holiday column check: PASSED


**Result**

The `school.holidays` object loaded successfully with 58,440 rows and 4 source columns. All expected fields are present, and the preview is consistent with the intended grain of one city–date record.

`Date` is currently stored as an object, while `schoolhols` and `school.hols` were imported as categorical fields from the original R object. Their data types will be standardised in Section 5.

All 9 expected official calendar PDFs are available. These documents remain unchanged as source evidence and will be used to structure the reviewed 2024–2026 holiday periods later.

No date conversion, missing-value treatment, or calendar-period extraction was performed in this section.

## 3. Prepare the Historical Match Table

### 3.1 Standardise Column Names and Data Types

### 3.2 Standardise Team and Venue Names

### 3.3 Apply the Reviewed 2024 Round Correlations

### 3.4 Create and Validate the Match Identifier

## 4. Prepare the 2026 Squiggle Match Snapshot

### 4.1 Select the Required Match and Fixture Fields

### 4.2 Apply Team and Venue Name Mappings

### 4.3 Standardise Match Dates and Status

### 4.4 Identify Completed Matches and Future Fixtures

### 4.5 Validate Record Uniqueness

## 5. Prepare the School-Holiday Table

### 5.1 Prepare the 2004-2023 Daily Holiday Records

### 5.2 Structure the Official 2024-2026 Calendar Periods

### 5.3 Expand Official Holiday Periods to Daily Records

### 5.4 Combine and Validate the Holiday Coverage

## 6. Validate the Prepared Tables

### 6.1 Validate Row Counts and Date Converage

### 6.2 Validate Missing Values and Duplicate Records

### 6.3 Validate Match Identifiers and Name Mappings

### 6.4 Summarise the Final Validation Results

## 7. Export and Summarise the Results

### 7.1 Export the Database-Ready Tables

### 7.2 Summarise the Output Tables

### 7.3 Document Limitations and Confirm the Next Step